# 05 — Expected loss

**Goal:** Turn the Cox default model into dollars. For each test loan, expected loss = chance of default within 36 months x loss given default x original balance. Compare the total with the losses that actually happened.

## Setup

The Cox model and the loss rate are both learned from training loans, using only what was known before Jan 2019.

In [ ]:
import pandas as pd

from mortgage_credit_risk.config import load_config
from mortgage_credit_risk.events import DEFAULT
from mortgage_credit_risk.loss import expected_loss, loan_outcomes, loss_given_default
from mortgage_credit_risk.macro import load_series
from mortgage_credit_risk.models import default_by, fit_cox
from mortgage_credit_risk.pipeline import build_model_tables

cfg = load_config()
H = cfg.loss_horizon_months
_, loans_path = build_model_tables(cfg, cfg.build_rate)
months_path = str(
    cfg.processed_dir / f"loan_months_{cfg.build_rate * 100:g}pct.parquet"
)
loans = pd.read_parquet(loans_path)
train = loans[loans["split"] == "train"]
test = loans[loans["split"] == "test"]
cox = fit_cox(train, cfg)

## Loss given default, from training

Where did training defaults stand by Jan 2019? Only defaults with a known outcome (ended, or current again) are used. Cured and paid-off loans count as no loss.

In [ ]:
defaults = train[train["event"] == DEFAULT]
outcomes = loan_outcomes(months_path, defaults["loan_id"], cfg.split.cutoff_period)
status = outcomes["zero_balance_code"].map(
    {"01": "paid off", "02": "credit loss", "03": "credit loss", "09": "credit loss"}
)
status = status.fillna(
    outcomes["last_delinquency"].map(
        lambda d: "current again" if d == "00" else "still late"
    )
).where(~outcomes["zero_balance_code"].isin(["15", "16", "96"]), "sold or removed")
print(status.value_counts(), "\n")

lgd = loss_given_default(defaults, outcomes)
print(f"Loss given default: {lgd:.2%} of original balance")

## Expected loss on test

Test loans are the 2019 vintage, followed to Mar 2026. Actual losses are from loans that defaulted within 36 months, counting losses up to Mar 2026.

In [ ]:
t = test.copy()
t["pd"] = default_by(cox, t, H)
t["expected_loss"] = expected_loss(t["pd"], lgd, t["orig_upb"])
t = t.join(loan_outcomes(months_path, t["loan_id"], 999912), on="loan_id")
defaulted = (t["event"] == DEFAULT) & (t["last_age"] <= H)

pd.DataFrame(
    {
        "predicted": {
            "defaults": t["pd"].sum(),
            "loss_given_default_pct": 100 * lgd,
            "loss_dollars": t["expected_loss"].sum(),
        },
        "actual": {
            "defaults": defaulted.sum(),
            "loss_given_default_pct": 100
            * t.loc[defaulted, "loss"].sum()
            / t.loc[defaulted, "orig_upb"].sum(),
            "loss_dollars": t.loc[defaulted, "loss"].sum(),
        },
    }
).round(2)

## Why test losses were smaller

House prices after the test loans were made (yearly average change, FRED Case-Shiller).

In [ ]:
prices = load_series("CSUSHPISA", cfg.fred_dir)
yearly = prices.groupby(prices.index.year).mean()
(100 * yearly.pct_change()).loc[2015:2023].round(1).rename("house_price_change_pct")

In [ ]:
late_now = t.loc[defaulted, "zero_balance_code"].isna() & (
    t.loc[defaulted, "last_delinquency"] != "00"
)
print(f"Test defaults still late in Mar 2026, loss not yet known: {late_now.sum()}")

## Findings

Numbers are for the 5% build sample. Test loans: 88,906 loans, $23 billion original balance.

1. **The loss rate from training is about 1% of the balance.** Of 884 training defaults, 319 were current again by Jan 2019, 174 had paid off, and only 42 had ended in a credit loss. Most 90+ day late loans do not cost the lender money. 342 were still late, so their outcome was unknown and they were left out.

2. **The model gets the number of defaults about right over 36 months.** It predicted 957 test defaults within 36 months, and 898 happened (ratio 1.07). This is much closer than the 12-month result in notebook 03 (0.72), where COVID hit the first year.

3. **But it predicts more than twice the losses that happened.** Predicted $2.25 million, actual $0.95 million. The whole gap is the loss rate: 0.97% predicted, 0.42% actual.

4. **House prices explain the smaller losses.** Prices rose 17% in 2021 and 15% in 2022. When a defaulted loan's home is worth much more than the loan, the lender can sell it with little or no loss. Training defaults came from calmer years (about 5% to 6% a year).

5. **Some test losses are not in yet.** 76 test defaults were still late in Mar 2026, so their final loss is unknown. Foreclosure pauses during COVID also slowed losses. The true actual loss is somewhat higher than $0.95 million, but not enough to close a gap of $1.3 million.

6. **The loss rate itself is probably too low.** It leaves out the 342 training defaults that were still late in Jan 2019, and those are the most likely to end in losses. Using a fixed loss rate learned in one housing market and applied to another is the weakest part of this estimate.

**Decision:**

- Keep the simple version: one loss rate from training, 36-month default chance from Cox, original balance.
- Report the result as it is. Default counts are close, and loss dollars are off because the housing market changed. A better loss model would use house prices after origination, which is beyond the scope of this project.